In [4]:
import pandas as pd
import numpy as np
import re

# 1. LOAD DATA
# Skipping the title row; handling the original Excel format
df_input = pd.read_excel('RMI_INPUT.xlsx', sheet_name='All HC Overall Scores', skiprows=1)
df_geo = pd.read_excel('Health Centers with Geocode.xlsx')

# 2. CLEANING & FILTERING
# Standardize headers
df_input.columns = [str(col).strip() for col in df_input.columns]

# --- REMOVE AVERAGE ROWS ---
# This filters out rows that contain the word "Average" in any column
df_input = df_input[~df_input.apply(lambda row: row.astype(str).str.contains('Average', case=False).any(), axis=1)]

# Forward fill Atoll/Island names for merged Excel cells
df_input['ATOLLS/ISLANDS (popn)'] = df_input['ATOLLS/ISLANDS (popn)'].ffill()

# 3. DATE STANDARDIZATION
def standardize_date(date_str):
    date_str = str(date_str).strip()
    if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
        return date_str
    # Extract first month and year for ranges
    m_match = re.findall(r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)', date_str, re.I)
    y_match = re.findall(r'(\d{4})', date_str)
    if m_match and y_match:
        try:
            return pd.to_datetime(f"{m_match[0]} 1 {y_match[0]}").strftime('%Y-%m-01')
        except: pass
    try:
        return pd.to_datetime(date_str).strftime('%Y-%m-01')
    except:
        return date_str

# 4. TRANSFORMATION: MELT (Pivoting columns to rows)
id_vars = ['ATOLLS/ISLANDS (popn)', 'HEALTH CENTERS (popn)*', 'Main vs Other HC', 
           'HEALTH ASSISTANTS', 'mh&cd Aide', 'Mayor']
date_cols = [c for c in df_input.columns if c not in id_vars and 'Unnamed' not in str(c)]

df_melted = df_input.melt(id_vars=id_vars, value_vars=date_cols, var_name='orig_date', value_name='score')
df_melted['date'] = df_melted['orig_date'].apply(standardize_date)

# 5. SCORE STANDARDIZATION (0-100 Scale)
def standardize_score(val):
    if pd.isna(val): return np.nan
    # Remove % and whitespace
    s = str(val).replace('%', '').strip()
    try:
        num = float(s)
        # Convert decimals (0-1 range) to percentage (0-100 range)
        # e.g., 0.59 -> 59.0
        if 0 < num < 1:
            return num * 100
        return num
    except:
        return np.nan

df_melted['score'] = df_melted['score'].apply(standardize_score)

# 6. GEOCODE MAPPING
df_geo[['latitude', 'longitude']] = df_geo['geocode_11'].str.split(',', expand=True)

def clean_name(val):
    if pd.isna(val): return ""
    return re.sub(r'\(.*\)', '', str(val)).strip().lower()

df_melted['hc_key'] = df_melted['HEALTH CENTERS (popn)*'].apply(clean_name)
df_geo['hc_key'] = df_geo['Health_facilities'].apply(clean_name)

df_out = pd.merge(df_melted, df_geo[['hc_key', 'latitude', 'longitude']], on='hc_key', how='left')

# 7. FINAL SCHEMA ALIGNMENT
output_mapping = {
    'ATOLLS/ISLANDS (popn)': 'island',
    'HEALTH CENTERS (popn)*': 'healt_centre',
    'Main vs Other HC': 'health_centre_type',
    'HEALTH ASSISTANTS': 'health_assistans',
    'mh&cd Aide': 'mhd_cd_aide',
    'Mayor': 'mayor'
}
df_out = df_out.rename(columns=output_mapping)

# Remove rows where score is missing after cleaning
df_out = df_out.dropna(subset=['score'])

# Uppercase and clean names
df_out['island'] = df_out['island'].apply(clean_name).str.upper()
df_out['healt_centre'] = df_out['healt_centre'].apply(clean_name).str.upper()

final_cols = ['island', 'healt_centre', 'latitude', 'longitude', 'health_centre_type', 
              'health_assistans', 'mhd_cd_aide', 'mayor', 'date', 'score']
df_out = df_out[final_cols]
df_out.insert(0, 'id', range(1, len(df_out) + 1))

# 8. SAVE
df_out.to_csv('RMI_OUTPUT.csv', index=False)
print(f"Final transformation successful: {len(df_out)} valid entries saved.")

Final transformation successful: 77 valid entries saved.
